In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import json
import time

root = Path("/kaggle/working")
repo = root / "rnn-hg-upstream"
logs = root / "rnn-hg-audit" / "commit1"
logs.mkdir(parents=True, exist_ok=True)

url = "https://github.com/RuiMao1988/Sequential-Metaphor-Identification.git"
commit = "eb924b57415f9d92d3b97df8f14881d5b200ff78"


def run(args, cwd=None, required=True):
    started = datetime.now(timezone.utc).isoformat()
    t0 = time.perf_counter()
    result = subprocess.run(
        args, cwd=cwd, capture_output=True, text=True
    )

    record = {
        "at": started,
        "command": args,
        "cwd": str(cwd) if cwd else None,
        "seconds": round(time.perf_counter() - t0, 3),
        "exit_code": result.returncode,
        "output": result.stdout + result.stderr
    }

    with (logs / "commands.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("$", " ".join(args))
    print(record["output"])

    if required:
        result.check_returncode()

    return result

# Lấy mã nguồn gốc
if not repo.exists():
    run(["git", "clone", "--no-checkout", url, str(repo)])
    run(["git", "checkout", "--detach", commit], cwd=repo)

# Kiểm tra commit gốc
actual = run(["git", "rev-parse", "HEAD"], cwd=repo).stdout.strip()
assert actual == commit, "Repository khác commit đã chọn."

diff = run(["git", "diff", "HEAD", "--"], cwd=repo).stdout
assert not diff, "File tác giả đã bị sửa."

(logs / "source.json").write_text(
    json.dumps({"repository": url, "commit": actual}, indent=2),
    encoding="utf-8"
)

print("SOURCE_OK")

$ git clone --no-checkout https://github.com/RuiMao1988/Sequential-Metaphor-Identification.git /kaggle/working/rnn-hg-upstream
Cloning into '/kaggle/working/rnn-hg-upstream'...

$ git checkout --detach eb924b57415f9d92d3b97df8f14881d5b200ff78
HEAD is now at eb924b5 Delete TroFi-X_formatted_svo.csv

$ git rev-parse HEAD
eb924b57415f9d92d3b97df8f14881d5b200ff78

$ git diff HEAD --

SOURCE_OK


In [5]:
import ast

# Đọc yêu cầu tác giả
readme = (repo / "README.md").read_text(encoding="utf-8")
requirements = (repo / "requirements.txt").read_text(encoding="utf-8")

print(readme)
print("REQUIREMENTS:\n", requirements)

# Trích cấu hình gốc
source = (repo / "rnn_hg/main_vua.py").read_text(encoding="utf-8")
tree = ast.parse(source)

names = {
    "r", "using_GPU", "batch_size",
    "num_epochs", "additional_epochs"
}
config = {}

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in names:
                config[target.id] = ast.literal_eval(node.value)

# Đọc kiến trúc gốc
for node in ast.walk(tree):
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        if node.func.id == "RNNSequenceModel":
            config["model"] = {
                item.arg: ast.literal_eval(item.value)
                for item in node.keywords
            }

# Đọc thiết lập train
for line in source.splitlines():
    if any(text in line for text in (
        "optim.SGD(",
        "class_weights =",
        "loss_criterion =",
        "cudnn.deterministic",
        "glove_embeddings =",
        "h5py.File("
    )):
        print(line.strip())

print(json.dumps(config, indent=2))

(logs / "original_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)
(logs / "original_requirements.txt").write_text(
    requirements, encoding="utf-8"
)

# Sequential-Metaphor-Identification
Code for the paper "End-to-End Sequential Metaphor Identification Inspired by Linguistic Theories"

https://www.aclweb.org/anthology/papers/P/P19/P19-1378/

## Embeddings
Download glove.840B.300d.zip (https://nlp.stanford.edu/projects/glove/) into glove folder.

Please email terry_mao0513@hotmail.com for ELMo embeddings, putting them into elmo folder.

(The used dataset and ELMo embeddings were cleaned and pre-trained by https://github.com/gao-g/metaphor-in-context.)

## Environment
python 3.6

pytorch 0.4.1
```
pip install -r requirements.txt
```
## Run
Go to the model folder (rnn_hg or rnn_mhca), then run for

1. VU Amsterdam Metaphor Corpus (VUA ALL POS and VUA VERB in the paper)
```
python main_vua.py
```
2. Mohammad dataset
```
python main_mohx.py
```
3. TroFi dataset
```
python main_trofi.py
```
## Citation
```
@inproceedings{mao2019metaphor,
  title={End-to-End Sequential Metaphor Identification Inspired by Linguistic Theories},
  author={Mao

201

In [6]:
import os
import sys
import shutil
import platform

# Kiểm tra máy chạy
hardware = {
    "at": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "ram_bytes": os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
    "disk_free_bytes": shutil.disk_usage(root).free
}

print(json.dumps(hardware, indent=2))

# Kiểm tra GPU hiện tại
if shutil.which("nvidia-smi"):
    gpu = run(["nvidia-smi"], required=False)
    hardware["nvidia_smi"] = gpu.stdout + gpu.stderr
else:
    hardware["nvidia_smi"] = "Không tìm thấy nvidia-smi."
    print(hardware["nvidia_smi"])

# Kiểm tra PyTorch notebook
probe = """
import torch
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("CUDA available:", torch.cuda.is_available())
"""

torch_check = run(
    [sys.executable, "-c", probe],
    required=False
)

hardware["torch_check_exit_code"] = torch_check.returncode
hardware["torch_output"] = torch_check.stdout + torch_check.stderr

(logs / "hardware.json").write_text(
    json.dumps(hardware, indent=2),
    encoding="utf-8"
)

{
  "at": "2026-11-16T07:25:39.322694+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "python_executable": "/usr/bin/python3",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "cpu_count": 4,
  "ram_bytes": 33658159104,
  "disk_free_bytes": 20873940992
}
$ nvidia-smi
Mon Nov 16 07:25:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4      

2519

In [7]:
import csv
import hashlib
from itertools import combinations

# Kiểm tra toàn dữ liệu
stats = {}
ids = {}

for split in ("train", "val", "test"):
    path = repo / "data/VUAsequence" / f"VUA_seq_formatted_{split}.csv"

    sentence_ids = set()
    sentences = tokens = positives = duplicates = 0

    with path.open(encoding="latin-1", newline="") as f:
        for row in csv.DictReader(f):
            words = row["sentence"].split()
            labels = ast.literal_eval(row["label_seq"])
            pos = ast.literal_eval(row["pos_seq"])

            assert len(words) == len(labels) == len(pos), (
                split, row["txt_id"], row["sen_ix"]
            )
            assert set(labels) <= {0, 1}, (
                split, row["txt_id"], row["sen_ix"]
            )

            key = (row["txt_id"], row["sen_ix"])
            duplicates += int(key in sentence_ids)
            sentence_ids.add(key)

            sentences += 1
            tokens += len(words)
            positives += sum(labels)

    ids[split] = sentence_ids
    stats[split] = {
        "sentences": sentences,
        "tokens": tokens,
        "metaphor_tokens": positives,
        "literal_tokens": tokens - positives,
        "metaphor_percent": round(100 * positives / tokens, 2),
        "duplicate_sentence_ids": duplicates,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest()
    }

# Kiểm tra giao ID
overlap = {
    f"{a}__{b}": len(ids[a] & ids[b])
    for a, b in combinations(ids, 2)
}

report = {
    "splits": stats,
    "sentence_id_overlap": overlap
}

(logs / "data_check.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print(json.dumps(report, indent=2))

# Đối chiếu dữ liệu
expected = {
    "train": (6323, 116622, 13051),
    "val": (1550, 38628, 4487),
    "test": (2694, 50175, 6243)
}

for split, counts in expected.items():
    found = stats[split]

    assert (
        found["sentences"],
        found["tokens"],
        found["metaphor_tokens"]
    ) == counts

    assert found["duplicate_sentence_ids"] == 0

assert all(count == 0 for count in overlap.values())

print("DATA_OK")
print("Nhật ký:", logs)

{
  "splits": {
    "train": {
      "sentences": 6323,
      "tokens": 116622,
      "metaphor_tokens": 13051,
      "literal_tokens": 103571,
      "metaphor_percent": 11.19,
      "duplicate_sentence_ids": 0,
      "sha256": "1c2db64a543f999626b765b0cf1f00d054a820b0ea005ccb536aaf1ff33f9b43"
    },
    "val": {
      "sentences": 1550,
      "tokens": 38628,
      "metaphor_tokens": 4487,
      "literal_tokens": 34141,
      "metaphor_percent": 11.62,
      "duplicate_sentence_ids": 0,
      "sha256": "66dc24219c2ba21d53d70ef0b0eac895f52cf18524238674cff4448e23feab3c"
    },
    "test": {
      "sentences": 2694,
      "tokens": 50175,
      "metaphor_tokens": 6243,
      "literal_tokens": 43932,
      "metaphor_percent": 12.44,
      "duplicate_sentence_ids": 0,
      "sha256": "33256891fb84cdda539def4781409762de6af645a415fbd13c7fcc08e5ee6135"
    }
  },
  "sentence_id_overlap": {
    "train__val": 0,
    "train__test": 0,
    "val__test": 0
  }
}
DATA_OK
Nhật ký: /kaggle/working/rnn

## Kết luận kiểm tra

- Mã nguồn đúng commit
- Train/dev/test có 6323/1550/2694 câu.
- Token, nhãn và POS khớp độ dài.
- Không trùng ID câu trong và giữa các tập.
- Kaggle: Python 3.13.16, PyTorch 2.11.0+cu130.
- GPU: hai Tesla T4, CUDA khả dụng.
- Tác giả yêu cầu Python 3.6, PyTorch 0.4.1.


In [10]:
import os

logs = root / "rnn-hg-audit" / "commit2"
logs.mkdir(parents=True, exist_ok=True)

mamba = root / "bin/micromamba"
original_env = Path("/tmp/rnn-hg-env-original")
t4_env = Path("/tmp/rnn-hg-env-t4")

os.environ["MAMBA_ROOT_PREFIX"] = "/tmp/rnn-hg-mamba"
os.environ["PIP_CACHE_DIR"] = "/tmp/rnn-hg-pip-cache"

# Chuẩn bị công cụ môi trường
if not mamba.exists():
    archive = root / "micromamba.tar.bz2"
    run([
        "curl", "-fL", "--retry", "3",
        "https://micro.mamba.pm/api/micromamba/linux-64/latest",
        "-o", str(archive)
    ])
    run([
        "tar", "-xjf", str(archive),
        "-C", str(root), "bin/micromamba"
    ])

# Chạy Python môi trường riêng
def env_python(prefix, *args, required=True):
    library_path = (
        str(prefix / "lib") + ":"
        + os.environ.get("LD_LIBRARY_PATH", "")
    )
    return run([
        "env",
        "CUDA_VISIBLE_DEVICES=0",
        "MPLBACKEND=Agg",
        "LD_LIBRARY_PATH=" + library_path,
        str(prefix / "bin/python"),
        *[str(arg) for arg in args]
    ], required=required)

if not (original_env / "bin/python").exists():
    run([
        str(mamba), "create", "-y",
        "-p", str(original_env),
        "--override-channels",
        "-c", "https://repo.anaconda.com/pkgs/main",
        "python=3.6.13", "pip", "cudatoolkit=9.0"
    ])

old_wheel = (
    "https://download.pytorch.org/whl/cu90/"
    "torch-0.4.1-cp36-cp36m-linux_x86_64.whl"
)

env_python(
    original_env, "-m", "pip", "install",
    old_wheel, "numpy==1.15.4"
)
env_python(original_env, "-m", "pip", "check")

$ curl -fL --retry 3 https://micro.mamba.pm/api/micromamba/linux-64/latest -o /kaggle/working/micromamba.tar.bz2
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100 7146k  100 7146k    0     0  9347k      0 --:--:-- --:--:-- --:--:-- 9347k

$ tar -xjf /kaggle/working/micromamba.tar.bz2 -C /kaggle/working bin/micromamba

$ /kaggle/working/bin/micromamba create -y -p /tmp/rnn-hg-env-original --override-channels -c https://repo.anaconda.com/pkgs/main python=3.6.13 pip cudatoolkit=9.0
⚠ Shard Index for pkgs/main/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/linux-64                                                      ✔ Starting
Using Flat Repodata for pkgs/main/linux-64             

CompletedProcess(args=['env', 'CUDA_VISIBLE_DEVICES=0', 'MPLBACKEND=Agg', 'LD_LIBRARY_PATH=/tmp/rnn-hg-env-original/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64', '/tmp/rnn-hg-env-original/bin/python', '-m', 'pip', 'check'], returncode=0, stdout='No broken requirements found.\n', stderr='')

In [11]:

probe = """
import sys
import json
import torch

torch.backends.cudnn.enabled = True

info = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cudnn_enabled": torch.backends.cudnn.enabled,
    "cuda_available": torch.cuda.is_available()
}
print(json.dumps(info, indent=2), flush=True)

assert torch.__version__ == sys.argv[1]
assert torch.version.cuda.startswith(sys.argv[2])
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1

gpu = torch.cuda.get_device_name(0)
print("GPU:", gpu, flush=True)
assert "T4" in gpu
print("ENV_READY", flush=True)

model = torch.nn.LSTM(
    input_size=1324,
    hidden_size=256,
    batch_first=True,
    bidirectional=True
).cuda()

x = torch.randn(2, 18, 1324).cuda()
y, _ = model(x)
loss = y.sum()
loss.backward()
torch.cuda.synchronize()

print("LSTM_GPU_OK", flush=True)
"""

probe_path = logs / "check_lstm.py"
probe_path.write_text(probe, encoding="utf-8")


before = env_python(
    original_env, "-u", probe_path,
    "0.4.1", "9.0",
    required=False
)

(logs / "lstm_before.txt").write_text(
    before.stdout + before.stderr,
    encoding="utf-8"
)

print("Exit code:", before.returncode)

$ env CUDA_VISIBLE_DEVICES=0 MPLBACKEND=Agg LD_LIBRARY_PATH=/tmp/rnn-hg-env-original/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64 /tmp/rnn-hg-env-original/bin/python -u /kaggle/working/rnn-hg-audit/commit2/check_lstm.py 0.4.1 9.0
{
  "python": "3.6.13 |Anaconda, Inc.| (default, Jun  4 2021, 14:25:59) \n[GCC 7.5.0]",
  "torch": "0.4.1",
  "cuda": "9.0.176",
  "cudnn": 7102,
  "cudnn_enabled": true,
  "cuda_available": true
}
GPU: Tesla T4
ENV_READY
Traceback (most recent call last):
  File "/kaggle/working/rnn-hg-audit/commit2/check_lstm.py", line 32, in <module>
    bidirectional=True
  File "/tmp/rnn-hg-env-original/lib/python3.6/site-packages/torch/nn/modules/module.py", line 258, in cuda
    return self._apply(lambda t: t.cuda(device))
  File "/tmp/rnn-hg-env-original/lib/python3.6/site-packages/torch/nn/modules/rnn.py", line 112, in _apply
    self.flatten_parameters()
  File "/tmp/rnn-hg-env-original/lib/pyth

In [12]:
# Kiểm tra lỗi thực tế
assert "ENV_READY" in before.stdout, (
    "Lỗi xảy ra trước phép thử LSTM. Dừng và kiểm tra output."
)
assert before.returncode != 0, (
    "Phép thử cũ không lỗi. Dừng để đối chiếu trước khi đổi môi trường."
)

# Tạo môi trường tương thích T4
if not (t4_env / "bin/python").exists():
    run([
        str(mamba), "create", "-y",
        "-p", str(t4_env),
        "--override-channels",
        "-c", "https://repo.anaconda.com/pkgs/main",
        "python=3.6.13", "pip",
        "cudatoolkit=10.0.130"
    ])

new_wheel = (
    "https://download.pytorch.org/whl/cu100/"
    "torch-1.0.0-cp36-cp36m-linux_x86_64.whl"
)

# Cài thư viện đường chạy
env_python(
    t4_env, "-m", "pip", "install",
    new_wheel,
    "numpy==1.15.4",
    "h5py==3.1.0",
    "tqdm==4.64.1",
    "matplotlib==3.3.4"
)

env_python(t4_env, "-m", "pip", "check")

$ /kaggle/working/bin/micromamba create -y -p /tmp/rnn-hg-env-t4 --override-channels -c https://repo.anaconda.com/pkgs/main python=3.6.13 pip cudatoolkit=10.0.130
⚠ Shard Index for pkgs/main/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/linux-64                                                      ✔ Starting
Using Flat Repodata for pkgs/main/linux-64                                                ✔ Done (0.2 sec)
⚠ Shard Index for pkgs/main/noarch not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/noarch                                                        ✔ Starting
Using Flat Repodata for pkgs/main/noarch                                                  ✔ Done (0.0 sec)
Parsing Packages' Records                                                                 ✔ Done (0.2 sec)

Resolving Environment                                                                           ⧖ Starting
Resolving Environment           

CompletedProcess(args=['env', 'CUDA_VISIBLE_DEVICES=0', 'MPLBACKEND=Agg', 'LD_LIBRARY_PATH=/tmp/rnn-hg-env-t4/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64', '/tmp/rnn-hg-env-t4/bin/python', '-m', 'pip', 'check'], returncode=0, stdout='No broken requirements found.\n', stderr='')

In [13]:
# Lưu kết quả sau sửa
after = env_python(
    t4_env, "-u", probe_path,
    "1.0.0", "10.0",
    required=False
)

(logs / "lstm_after.txt").write_text(
    after.stdout + after.stderr,
    encoding="utf-8"
)

assert after.returncode == 0, "Phép thử chưa đạt. Xem output."
assert "LSTM_GPU_OK" in after.stdout

print("LSTM_GPU_OK")

$ env CUDA_VISIBLE_DEVICES=0 MPLBACKEND=Agg LD_LIBRARY_PATH=/tmp/rnn-hg-env-t4/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64 /tmp/rnn-hg-env-t4/bin/python -u /kaggle/working/rnn-hg-audit/commit2/check_lstm.py 1.0.0 10.0
{
  "python": "3.6.13 |Anaconda, Inc.| (default, Jun  4 2021, 14:25:59) \n[GCC 7.5.0]",
  "torch": "1.0.0",
  "cuda": "10.0.130",
  "cudnn": 7401,
  "cudnn_enabled": true,
  "cuda_available": true
}
GPU: Tesla T4
ENV_READY
LSTM_GPU_OK

LSTM_GPU_OK


In [14]:
# Lưu phiên bản thực tế
for name, prefix in (
    ("original", original_env),
    ("t4", t4_env)
):
    frozen = env_python(prefix, "-m", "pip", "freeze")
    (logs / f"{name}_pip_freeze.txt").write_text(
        frozen.stdout, encoding="utf-8"
    )

    packages = run([
        str(mamba), "list",
        "-p", str(prefix), "--explicit"
    ])
    (logs / f"{name}_conda_explicit.txt").write_text(
        packages.stdout, encoding="utf-8"
    )

# Ghi lỗi thực tế
error_lines = before.stderr.strip().splitlines()
error = error_lines[-1] if error_lines else before.stdout.strip()

summary = {
    "before": {
        "environment": str(original_env),
        "wheel": old_wheel,
        "exit_code": before.returncode,
        "error": error
    },
    "after": {
        "environment": str(t4_env),
        "wheel": new_wheel,
        "exit_code": after.returncode,
        "lstm_gpu_ok": "LSTM_GPU_OK" in after.stdout
    },
    "probe": str(probe_path),
    "visible_gpu": "0",
    "training_started": False
}

(logs / "environment_comparison.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Kiểm tra nguồn chưa sửa
diff = run(["git", "diff", "HEAD", "--"], cwd=repo).stdout
assert not diff, "File tác giả đã bị sửa."

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Nhật ký:", logs)

$ env CUDA_VISIBLE_DEVICES=0 MPLBACKEND=Agg LD_LIBRARY_PATH=/tmp/rnn-hg-env-original/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64 /tmp/rnn-hg-env-original/bin/python -m pip freeze
certifi==2021.5.30
numpy==1.15.4
torch @ https://download.pytorch.org/whl/cu90/torch-0.4.1-cp36-cp36m-linux_x86_64.whl

$ /kaggle/working/bin/micromamba list -p /tmp/rnn-hg-env-original --explicit
List of packages in environment: "/tmp/rnn-hg-env-original"

https://repo.anaconda.com/pkgs/main/linux-64/_libgcc_mutex-0.1-main.conda
https://repo.anaconda.com/pkgs/main/linux-64/_openmp_mutex-5.1-52_gnu.conda
https://repo.anaconda.com/pkgs/main/noarch/ca-certificates-2026.11.5-h59cdc77_0.conda
https://repo.anaconda.com/pkgs/main/linux-64/certifi-2021.5.30-py36h06a4308_0.conda
https://repo.anaconda.com/pkgs/main/linux-64/cudatoolkit-9.0-h13b8566_0.conda
https://repo.anaconda.com/pkgs/main/linux-64/ld_impl_linux-64-2.44-h9e0c5a2_3.conda
https:/

## Xử lý tương thích GPU

- Python 3.6.13, PyTorch 0.4.1, CUDA 9.0 và cuDNN 7.1.2 nhận được GPU T4 nhưng phép thử LSTM thất bại.
- Lỗi: RuntimeError: CuDNN error: CUDNN_STATUS_SUCCESS.
- Chuyển sang Python 3.6.13, PyTorch 1.0.0 và CUDA 10.0.
- Chạy lại cùng phép thử: LSTM_GPU_OK, exit code 0.
- Forward và backward chạy trên một T4, cuDNN bật.

In [15]:
import shutil
from IPython.display import FileLink, display

shutil.make_archive(
    "/kaggle/working/commit2_logs",
    "zip",
    "/kaggle/working/rnn-hg-audit/commit2"
)

display(FileLink("/kaggle/working/commit2_logs.zip"))

/kaggle/working/commit2_logs.zip

In [16]:
import sys
import shutil
import zipfile
import hashlib

logs = root / "rnn-hg-audit" / "commit3"
downloads = root / "rnn-hg-downloads"

logs.mkdir(parents=True, exist_ok=True)
downloads.mkdir(exist_ok=True)

glove_url = (
    "https://huggingface.co/stanfordnlp/glove/"
    "resolve/main/glove.840B.300d.zip"
)
elmo_id = "1QdtX-sg73ALRDUmjbTtyai_ZLOem_DsX"
elmo_url = f"https://drive.google.com/file/d/{elmo_id}/view"

sources = {
    "glove": glove_url,
    "elmo": elmo_url,
    "elmo_reference": "https://github.com/gao-g/metaphor-in-context"
}

(logs / "resource_sources.json").write_text(
    json.dumps(sources, indent=2),
    encoding="utf-8"
)

# Giải nén đúng tài nguyên
def extract_file(archive, filename, destination):
    if destination.is_file() and destination.stat().st_size > 0:
        print("Đã có:", destination)
        return

    with zipfile.ZipFile(archive) as z:
        matches = [
            item for item in z.infolist()
            if not item.is_dir()
            and Path(item.filename).name == filename
        ]
        assert len(matches) == 1, f"Không tìm thấy duy nhất {filename}."

        item = matches[0]
        assert shutil.disk_usage(root).free > item.file_size, (
            "Không đủ dung lượng giải nén."
        )

        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = Path(str(destination) + ".part")

        with z.open(item) as src, partial.open("wb") as dst:
            shutil.copyfileobj(src, dst)

        partial.replace(destination)
        print("Đã giải nén:", destination)

print("Dung lượng trống: %.2f GiB" % (
    shutil.disk_usage(root).free / 1024**3
))

Dung lượng trống: 19.42 GiB


In [17]:
glove_zip = downloads / "glove.840B.300d.zip"
glove_file = repo / "glove/glove840B300d.txt"

# Tải khi chưa có
if not glove_file.is_file():
    if not glove_zip.is_file() or not zipfile.is_zipfile(glove_zip):
        partial = Path(str(glove_zip) + ".part")

        run([
            "curl", "-fL", "--retry", "3",
            glove_url, "-o", str(partial)
        ])

        assert zipfile.is_zipfile(partial), "GloVe tải về không phải ZIP."
        partial.replace(glove_zip)

    extract_file(
        glove_zip,
        "glove.840B.300d.txt",
        glove_file
    )

assert glove_file.is_file() and glove_file.stat().st_size > 0
print("GLOVE_FILE_OK")

$ curl -fL --retry 3 https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip -o /kaggle/working/rnn-hg-downloads/glove.840B.300d.zip.part
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   994  100   994    0     0  15968      0 --:--:-- --:--:-- --:--:-- 16032

  0 2075M    0 13880    0     0  16490      0 36:40:05 --:--:-- 36:40:05 16490
  6 2075M    6  135M    0     0  64.9M      0  0:00:31  0:00:02  0:00:29  108M
 10 2075M   10  207M    0     0  75.5M      0  0:00:27  0:00:02  0:00:25  108M
 15 2075M   15  327M    0     0  86.7M      0  0:00:23  0:00:03  0:00:20  111M
 22 2075M   22  457M    0     0  97.4M      0  0:00:21  0:00:04  0:00:17  118M
 27 2075M   27  567M    0     0  99.5M      0  0:00:20  0:00:05  0:00:15  116M
 33 2075M   33  703M    0     0   102M      0  0:00:

In [18]:
tools = Path("/tmp/rnn-hg-download-tools")
elmo_zip = downloads / "elmo.zip"

elmo_files = [
    repo / "elmo" / f"VUA_{split}.hdf5"
    for split in ("train", "val", "test")
]

# Tải khi còn thiếu
if not all(path.is_file() for path in elmo_files):
    if not elmo_zip.is_file() or not zipfile.is_zipfile(elmo_zip):

        # Cài công cụ tải riêng
        if not (tools / "gdown").is_dir():
            run([
                sys.executable, "-m", "pip", "install",
                "--target", str(tools),
                "gdown==6.5.1"
            ])

        downloader = """
import sys
sys.path.insert(0, sys.argv[1])
import gdown

result = gdown.download(
    id=sys.argv[2],
    output=sys.argv[3],
    quiet=False
)
if result is None:
    raise RuntimeError("Khong tai duoc ELMo.")
"""

        partial = Path(str(elmo_zip) + ".part")

        run([
            sys.executable, "-c", downloader,
            str(tools), elmo_id, str(partial)
        ])

        assert zipfile.is_zipfile(partial), "ELMo tải về không phải ZIP."
        partial.replace(elmo_zip)

    for path in elmo_files:
        extract_file(elmo_zip, path.name, path)

assert all(path.is_file() for path in elmo_files)
print("ELMO_FILES_OK")

$ /usr/bin/python3 -m pip install --target /tmp/rnn-hg-download-tools gdown==6.5.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.

$ /usr/bin/python3 -c 
import sys
sys.path.insert(0, sys.argv[1])
i

In [19]:
check_resources = r"""
import ast
import csv
import json
import sys
from pathlib import Path
import h5py
import numpy as np

repo = Path(sys.argv[1])
logs = Path(sys.argv[2])

# Đọc từ vựng train
vocab = set()
train_path = repo / "data/VUAsequence/VUA_seq_formatted_train.csv"

with train_path.open(encoding="latin-1", newline="") as f:
    for row in csv.DictReader(f):
        vocab.update(row["sentence"].split())

# Kiểm tra vector GloVe
matched = set()
glove_path = repo / "glove/glove840B300d.txt"

with glove_path.open(encoding="utf-8") as f:
    for line in f:
        fields = line.rstrip().split()

        if len(fields) != 301 or fields[0] not in vocab:
            continue

        vector = np.asarray(fields[1:], dtype=np.float32)
        assert vector.shape == (300,)
        assert np.isfinite(vector).all()
        matched.add(fields[0])

assert matched, "Khong tim thay GloVe cho tu vung train."

report = {
    "glove": {
        "dimension": 300,
        "train_vocabulary": len(vocab),
        "matched_words": len(matched),
        "unmatched_words": len(vocab - matched)
    },
    "vua": {}
}

# Kiểm tra mọi câu VUA
for split in ("train", "val", "test"):
    data_path = repo / "data/VUAsequence" / (
        "VUA_seq_formatted_" + split + ".csv"
    )
    elmo_path = repo / "elmo" / ("VUA_" + split + ".hdf5")

    sentences = tokens = metaphors = 0

    with h5py.File(str(elmo_path), "r") as embeddings:
        with data_path.open(encoding="latin-1", newline="") as f:
            for row in csv.DictReader(f):
                sentence = row["sentence"]
                words = sentence.split()
                labels = ast.literal_eval(row["label_seq"])
                pos = ast.literal_eval(row["pos_seq"])

                assert len(words) == len(labels) == len(pos)
                assert set(labels) <= {0, 1}
                assert sentence in embeddings, (
                    "Missing ELMo", split, row["txt_id"], row["sen_ix"]
                )
                assert embeddings[sentence].shape == (len(words), 1024), (
                    "Wrong ELMo shape", split, row["txt_id"], row["sen_ix"]
                )

                sentences += 1
                tokens += len(words)
                metaphors += sum(labels)

    report["vua"][split] = {
        "sentences": sentences,
        "tokens": tokens,
        "metaphor_tokens": metaphors,
        "elmo_dimension": 1024
    }

# Đối chiếu đầy đủ dữ liệu
expected = {
    "train": (6323, 116622, 13051),
    "val": (1550, 38628, 4487),
    "test": (2694, 50175, 6243)
}

for split, counts in expected.items():
    result = report["vua"][split]
    assert (
        result["sentences"],
        result["tokens"],
        result["metaphor_tokens"]
    ) == counts

(logs / "resource_check.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print(json.dumps(report, indent=2))
print("FULL_RESOURCE_OK")
"""

check_path = logs / "check_resources.py"
check_path.write_text(check_resources, encoding="utf-8")

env_python(
    t4_env, "-u", check_path,
    str(repo), str(logs)
)

$ env CUDA_VISIBLE_DEVICES=0 MPLBACKEND=Agg LD_LIBRARY_PATH=/tmp/rnn-hg-env-t4/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64 /tmp/rnn-hg-env-t4/bin/python -u /kaggle/working/rnn-hg-audit/commit3/check_resources.py /kaggle/working/rnn-hg-upstream /kaggle/working/rnn-hg-audit/commit3
{
  "glove": {
    "dimension": 300,
    "train_vocabulary": 13843,
    "matched_words": 13404,
    "unmatched_words": 439
  },
  "vua": {
    "train": {
      "sentences": 6323,
      "tokens": 116622,
      "metaphor_tokens": 13051,
      "elmo_dimension": 1024
    },
    "val": {
      "sentences": 1550,
      "tokens": 38628,
      "metaphor_tokens": 4487,
      "elmo_dimension": 1024
    },
    "test": {
      "sentences": 2694,
      "tokens": 50175,
      "metaphor_tokens": 6243,
      "elmo_dimension": 1024
    }
  }
}
FULL_RESOURCE_OK



CompletedProcess(args=['env', 'CUDA_VISIBLE_DEVICES=0', 'MPLBACKEND=Agg', 'LD_LIBRARY_PATH=/tmp/rnn-hg-env-t4/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64', '/tmp/rnn-hg-env-t4/bin/python', '-u', '/kaggle/working/rnn-hg-audit/commit3/check_resources.py', '/kaggle/working/rnn-hg-upstream', '/kaggle/working/rnn-hg-audit/commit3'], returncode=0, stdout='{\n  "glove": {\n    "dimension": 300,\n    "train_vocabulary": 13843,\n    "matched_words": 13404,\n    "unmatched_words": 439\n  },\n  "vua": {\n    "train": {\n      "sentences": 6323,\n      "tokens": 116622,\n      "metaphor_tokens": 13051,\n      "elmo_dimension": 1024\n    },\n    "val": {\n      "sentences": 1550,\n      "tokens": 38628,\n      "metaphor_tokens": 4487,\n      "elmo_dimension": 1024\n    },\n    "test": {\n      "sentences": 2694,\n      "tokens": 50175,\n      "metaphor_tokens": 6243,\n      "elmo_dimension": 1024\n    }\n  }\n}\nFULL_RESOURCE

In [20]:
# Tính mã kiểm tra file
def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

# Ghi tài nguyên thực tế
manifest = {}

for path in [glove_file] + elmo_files:
    manifest[str(path.relative_to(repo))] = {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path)
    }

(logs / "resource_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print(json.dumps(manifest, indent=2))

# Kiểm tra nguồn chưa sửa
diff = run(["git", "diff", "HEAD", "--"], cwd=repo).stdout
assert not diff, "File tác giả đã bị sửa."

print("RESOURCES_READY")
print("Nhật ký:", logs)

{
  "glove/glove840B300d.txt": {
    "bytes": 5646239124,
    "sha256": "3546fe00b5659a045c7fd9879b7a70f02985205c43f9b90bf9ea1434adc38db2"
  },
  "elmo/VUA_train.hdf5": {
    "bytes": 623425655,
    "sha256": "1da3fb6778dc4c330ab755bfe1843789182ea70f4cd7d1a90c5454decee8817e"
  },
  "elmo/VUA_val.hdf5": {
    "bytes": 158385664,
    "sha256": "8af69bbf02d362d952a6684f4b6780eeb1e8137332d39de0d23a30ceaab014ea"
  },
  "elmo/VUA_test.hdf5": {
    "bytes": 206446613,
    "sha256": "654fd41f8a775f151b7ee1a4281181dc7c6693423bad49040ddb048ffb29726f"
  }
}
$ git diff HEAD --

RESOURCES_READY
Nhật ký: /kaggle/working/rnn-hg-audit/commit3


## Kết luận tài nguyên

- GloVe: 300 chiều, khớp 13.404/13.843 từ train.
- Có 439 từ không khớp GloVe.
- Mọi câu train/dev/test có ELMo đúng kích thước 1024 chiều.
- Số câu và token khớp dữ liệu đã kiểm tra.
- Đã lưu nguồn tải, kích thước và SHA-256.
